# Exercises: NLP Fundamentals

Build core NLP components from scratch and apply them to text tasks.

## Exercise 1: Custom Tokenizer

Build a **word-level tokenizer** from scratch.

**Requirements:**
- `fit(corpus)` — build vocabulary from a list of strings
- `encode(text)` → list of integer IDs (with `<UNK>` for unseen words)
- `decode(ids)` → reconstructed string
- Support `max_vocab_size` (keep most frequent words)
- Test on a small corpus

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import re
from collections import Counter


class SimpleTokenizer:
    def __init__(self, max_vocab_size=1000):
        self.max_vocab_size = max_vocab_size
        self.word2id = {}
        self.id2word = {}
        self.PAD, self.UNK = 0, 1

    def _tokenize(self, text):
        return re.findall(r"\b\w+\b", text.lower())

    def fit(self, corpus):
        counts = Counter()
        for doc in corpus:
            counts.update(self._tokenize(doc))
        most_common = counts.most_common(self.max_vocab_size - 2)
        self.word2id = {'<PAD>': self.PAD, '<UNK>': self.UNK}
        for idx, (word, _) in enumerate(most_common, start=2):
            self.word2id[word] = idx
        self.id2word = {v: k for k, v in self.word2id.items()}
        return self

    def encode(self, text):
        return [self.word2id.get(w, self.UNK) for w in self._tokenize(text)]

    def decode(self, ids):
        return ' '.join(self.id2word.get(i, '<UNK>') for i in ids)


corpus = [
    "The quick brown fox jumps over the lazy dog",
    "A quick brown dog outpaces the fox",
    "The fox and the dog became friends",
    "Machine learning is a branch of artificial intelligence",
    "Deep learning uses neural networks with many layers",
]

tok = SimpleTokenizer(max_vocab_size=30)
tok.fit(corpus)

test = "The quick fox uses deep learning"
ids = tok.encode(test)
decoded = tok.decode(ids)

print(f"Vocab size: {len(tok.word2id)}")
print(f"Input:   '{test}'")
print(f"Encoded: {ids}")
print(f"Decoded: '{decoded}'")
print(f"\nVocab: {list(tok.word2id.keys())}")


### Explanation

The tokenizer uses regex-based word splitting and a frequency-based vocabulary. Words not in the vocabulary map to `<UNK>` (ID 1). This is the foundation of all neural NLP — converting text to integer sequences that embedding layers can consume.

## Exercise 2: TF-IDF from Scratch

Implement **TF-IDF vectorisation** without using sklearn's `TfidfVectorizer`.

**Requirements:**
- Compute term frequency (TF) per document
- Compute inverse document frequency (IDF) across the corpus
- Multiply to get TF-IDF matrix
- Compare your output with `sklearn.feature_extraction.text.TfidfVectorizer`

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are friends",
    "the mat is on the floor",
]


def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


# Build vocabulary
all_tokens = set()
for doc in corpus:
    all_tokens.update(tokenize(doc))
vocab = sorted(all_tokens)
word2idx = {w: i for i, w in enumerate(vocab)}
n_docs, n_vocab = len(corpus), len(vocab)

# TF: count / total words in doc
tf = np.zeros((n_docs, n_vocab))
for d, doc in enumerate(corpus):
    tokens = tokenize(doc)
    counts = Counter(tokens)
    for word, count in counts.items():
        tf[d, word2idx[word]] = count / len(tokens)

# IDF: log(N / df) + 1  (smooth variant)
df = np.sum(tf > 0, axis=0)
idf = np.log(n_docs / (df + 1)) + 1  # smoothed IDF

# TF-IDF
tfidf = tf * idf

# L2 normalise rows
norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
norms[norms == 0] = 1
tfidf_norm = tfidf / norms

print("Custom TF-IDF (first doc, non-zero):")
nonzero = np.where(tfidf_norm[0] > 0)[0]
for idx in nonzero:
    print(f"  {vocab[idx]:>10}: {tfidf_norm[0, idx]:.4f}")

# Sklearn comparison
sk = TfidfVectorizer()
sk_matrix = sk.fit_transform(corpus).toarray()
print(f"\nSklearn shape: {sk_matrix.shape}")
print(f"Custom  shape: {tfidf_norm.shape}")


### Explanation

TF measures word importance within a document; IDF measures rarity across the corpus. Their product up-weights terms that are frequent in a document but rare overall (i.e., discriminative). L2 normalisation makes documents comparable regardless of length. Minor numerical differences vs sklearn come from different smoothing constants.

## Exercise 3: Multi-Class Text Classification

Classify synthetic text snippets into 4 categories.

**Requirements:**
- Generate synthetic labelled text data (≥200 samples)
- Use `TfidfVectorizer` → classifier pipeline
- Try at least 2 classifiers (e.g. MultinomialNB, LinearSVC)
- Print classification report for each

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Synthetic corpus with category-specific keywords
templates = {
    'sports': [
        "The team won the {adj} championship game",
        "The player scored a {adj} goal in the match",
        "Athletes competed in the {adj} tournament",
        "The coach praised the {adj} performance on the field",
    ],
    'tech': [
        "The new {adj} software update improves performance",
        "Engineers developed a {adj} algorithm for processing",
        "The {adj} processor runs faster than competitors",
        "Cloud computing enables {adj} scalable solutions",
    ],
    'politics': [
        "The senator proposed a {adj} new policy today",
        "Voters are concerned about {adj} economic reforms",
        "The {adj} election results surprised analysts",
        "Government announced {adj} new regulations",
    ],
    'science': [
        "Researchers discovered a {adj} new species",
        "The {adj} experiment confirmed the hypothesis",
        "Scientists published {adj} findings in the journal",
        "The {adj} study reveals new insights about climate",
    ],
}
adjectives = ['remarkable', 'incredible', 'surprising', 'important', 'major',
              'significant', 'impressive', 'critical', 'outstanding', 'key']

texts, labels = [], []
for label, tmpls in templates.items():
    for _ in range(50):
        t = np.random.choice(tmpls).format(adj=np.random.choice(adjectives))
        texts.append(t)
        labels.append(label)

texts, labels = np.array(texts), np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels)

classifiers = {
    'MultinomialNB': MultinomialNB(),
    'LinearSVC': LinearSVC(random_state=42, max_iter=2000),
}

for name, clf in classifiers.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
        ('clf', clf),
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    cv = cross_val_score(pipe, texts, labels, cv=5, scoring='accuracy')
    print(f"=== {name} (CV acc: {cv.mean():.4f}) ===")
    print(classification_report(y_test, preds))


### Explanation

TF-IDF + linear classifiers is a strong text classification baseline. `ngram_range=(1,2)` captures bigrams which help with phrases like 'new policy' vs 'new species'. LinearSVC often outperforms Naive Bayes when classes are well-separated in TF-IDF space.

## Exercise 4: Text Preprocessing Pipeline

Build a **reusable text preprocessing pipeline** (as a class) with:
1. Lowercasing
2. URL and email removal
3. Punctuation removal
4. Number normalisation (digits → `<NUM>`)
5. Whitespace collapsing
6. Optional stopword removal

**Requirements:**
- Each step should be toggleable
- Process a list of messy text samples and show before/after

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import re


class TextPreprocessor:
    STOPWORDS = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'be', 'been',
                 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
                 'would', 'could', 'should', 'may', 'might', 'can', 'shall',
                 'to', 'of', 'in', 'for', 'on', 'with', 'at', 'by', 'from',
                 'and', 'or', 'but', 'not', 'no', 'if', 'it', 'its', 'this',
                 'that', 'so', 'as'}

    def __init__(self, lowercase=True, remove_urls=True, remove_emails=True,
                 remove_punctuation=True, normalize_numbers=True,
                 remove_stopwords=False):
        self.lowercase = lowercase
        self.remove_urls = remove_urls
        self.remove_emails = remove_emails
        self.remove_punctuation = remove_punctuation
        self.normalize_numbers = normalize_numbers
        self.remove_stopwords = remove_stopwords

    def process(self, text):
        if self.lowercase:
            text = text.lower()
        if self.remove_urls:
            text = re.sub(r'https?://\S+|www\.\S+', '', text)
        if self.remove_emails:
            text = re.sub(r'\S+@\S+\.\S+', '', text)
        if self.remove_punctuation:
            text = re.sub(r'[^\w\s]', ' ', text)
        if self.normalize_numbers:
            text = re.sub(r'\b\d+\b', '<NUM>', text)
        text = re.sub(r'\s+', ' ', text).strip()  # collapse whitespace
        if self.remove_stopwords:
            tokens = text.split()
            tokens = [t for t in tokens if t not in self.STOPWORDS]
            text = ' '.join(tokens)
        return text

    def batch_process(self, texts):
        return [self.process(t) for t in texts]


messy_texts = [
    "Check out https://example.com for MORE info!!!",
    "Contact us at support@company.com or call 1-800-555-1234",
    "   The   Quick  BROWN fox  jumped  over 3 lazy dogs!!!  ",
    "RT @user: Machine Learning in 2024 is AMAZING #AI #ML",
    "Price: $19.99 — visit www.shop.com for 50% off!!",
]

pp = TextPreprocessor(remove_stopwords=True)

print("Before → After\n" + "=" * 60)
for raw in messy_texts:
    clean = pp.process(raw)
    print(f"  IN:  {raw}")
    print(f"  OUT: {clean}\n")


### Explanation

Preprocessing is applied in a fixed order — lowercasing first so subsequent regex patterns only need to handle one case. Each step is togglable so the pipeline can be adapted per task (e.g., keep numbers for financial text). Stopword removal is optional because modern models (TF-IDF with sublinear TF, or transformers) often handle stopwords implicitly.